[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/therestiscode/notebooks/blob/main/03-float32-ceiling/ceiling.ipynb)

# I summed twenty million ones. I got 16,777,216.

A table of ones, two columns, twenty million rows. Each column should add up to
20,000,000. In `float32`, NumPy says 16,777,216, and it says it without a
warning.

**The first eight code cells, `In [1]` to `In [8]`, are the ones in the short**, in
the same order, printing the same output. Run them top to bottom. After them, the
rest of the evidence: the warnings switched on, the running total watched as it
stalls, NumPy's own documentation, and a real-looking sales table.

The arrays are about 160 MB each, which is fine on a free Colab runtime.

## Add one to this number

In [1]:
import numpy as np
wall = np.float32(16_777_216)
print(float(wall + 1))

16777216.0


One added to 16,777,216 comes back as 16,777,216. No error, no warning.

`float()` only turns NumPy's `float32` into a plain Python number, exactly, so it
prints as `16777216.0`. A bare `print(wall + 1)` would show `1.6777216e+07`, the
same value in scientific notation.

## Twenty million ones, two columns

In [2]:
N = 20_000_000
table = np.ones((N, 2), dtype="float32")
print(table[:3])

[[1. 1.]
 [1. 1.]
 [1. 1.]]


In [3]:
print(table.sum(axis=0).tolist())

[16777216.0, 16777216.0]


Ones add up to the number of rows, so both totals should be 20,000,000.
Both are 16,777,216.

## Delete the other column, run the same line

In [4]:
alone = np.delete(table, 1, axis=1)
print(alone.sum(axis=0).tolist())

[20000000.0]


Same ones, same `.sum(axis=0)`. The only change is that the column lost its
neighbour, and now the total is right.

## The whole numbers `float32` has

16,777,216 is 2²⁴. A `float32` keeps 24 bits of precision, so up to that number
it stores every whole number. Past it, only the even ones. Ask for an odd one and
you get an even neighbour:

In [5]:
print(float(np.float32(16_777_217)))
print(float(np.float32(16_777_219)))

16777216.0
16777220.0


## Why two columns hit it and one column doesn't

The table is stored row by row (C order), so the two values in a row sit next to
each other in memory. To total each column, NumPy walks down the rows and adds
each value into a running total: one, then two, then three. That running total
reaches 16,777,216, and every one added after that rounds back.

A single column is one unbroken run of memory, and there NumPy adds the numbers
in pairs, then pairs of pairs (pairwise summation). A one only ever meets a small
number, so nothing is lost. Both routes are shown step by step after `In [8]`.

## One column was luck, not a cure

Add one more row:

In [6]:
N = 20_000_001
alone = np.ones((N, 1), dtype="float32")
print(alone.sum(axis=0).tolist())

[20000000.0]


20,000,001 ones, and it still says 20,000,000. On one column the total does pass
16,777,216, at the very last step, when the two big halves are added. It landed on
20,000,000 only because 20,000,000 is even.

## The fix: do the total in `float64`

In [7]:
total = table.sum(axis=0,
                  dtype="float64")
print(total.tolist())

[20000000.0, 20000000.0]


The data stays `float32`, so you keep the memory you chose it for. Only the total
is kept in `float64`, which has room for whole numbers up to 2⁵³.

## Twenty million fits in `float32`

The number itself was never the problem:

In [8]:
print(float(np.float32(20_000_000)))

20000000.0


One at a time, the total never gets there. Keep your data in `float32` if you
want, but do the total in `float64`.

---

# The rest of the evidence

The short's cells end above. Everything below is for checking it.

## No warning, even when NumPy is told to be loud

Turn every warning into an exception, and every floating-point error NumPy checks
for too, then run the same sum:

In [9]:
import warnings

warnings.simplefilter("error")
np.seterr(all="raise")
print(table.sum(axis=0).tolist())

[16777216.0, 16777216.0]


It still prints 16,777,216 with no exception. Adding one to 16,777,216 rounds back
to 16,777,216: that is the IEEE rounding rule doing its job, so there is nothing
to warn about.

## How many ones went missing

`N` is 20,000,001 since `In [6]`, so set it back first:

In [10]:
N = 20_000_000
lost = N - int(table.sum(axis=0)[0])
print(f"{lost:,}")

3,222,784


## The gaps around 16,777,216

In [11]:
print(float(wall - 1))
print(float(wall + 2))
print(float(np.spacing(wall)))

16777215.0
16777218.0
2.0


Below it, whole numbers sit 1 apart. Above it, `np.spacing` says the next number
`float32` can hold is 2 away.

## What the neighbour holds doesn't matter

It only has to be there:

In [12]:
ones = np.ones(N, dtype="float32")
zeros = np.zeros(N, dtype="float32")
print(np.column_stack([ones, zeros]).sum(axis=0).tolist())
print(np.ones((N, 3), dtype="float32").sum(axis=0).tolist())

[16777216.0, 0.0]
[16777216.0, 16777216.0, 16777216.0]


## Watch the running total stall

`np.cumsum` builds exactly the running total two columns get, so you can watch it
stop:

In [13]:
running = np.cumsum(ones)
print(f"{float(running[16_777_215]):,.0f}")
print(f"{float(running[16_777_216]):,.0f}")
print(f"{float(running[-1]):,.0f}")
del running

16,777,216
16,777,216
16,777,216


After 16,777,216 ones, after 16,777,217, and after all 20,000,000: the same number.

## The pairs: half plus half

NumPy's real blocks are smaller than halves, but the idea is the same: no big
total ever meets a lone one.

In [14]:
half = np.ones(N // 2, dtype="float32").sum()
print(float(half))
print(float(half + half))

10000000.0
20000000.0


## NumPy says so, if you know where to look

In [15]:
doc = " ".join(np.sum.__doc__.split())
start = doc.index("This improved precision is always provided")
end = doc.index("fast axis in memory.") + len("fast axis in memory.")
print(doc[start:end])

This improved precision is always provided when no ``axis`` is given. When ``axis`` is given, it will depend on which axis is summed. Technically, to provide the best speed possible, the improved precision is only used when the summation is along the fast axis in memory.


"The fast axis in memory" is the one whose values sit next to each other. In a
two-column C-order table that is the row, not the column you are totalling.

## "Who has twenty million ones?"

Nobody. The ones only make the arithmetic easy to check. Here are twenty million
prices between 1.00 and 100.00, with a quantity column beside them, which is an
ordinary sales table:

In [16]:
del table, alone, ones, zeros
rng = np.random.default_rng(13)
price = np.round(rng.uniform(1, 100, N), 2).astype(np.float32)
qty = rng.integers(1, 5, N).astype(np.float32)
sales = np.column_stack([price, qty])

true = float(price.sum(dtype=np.float64))
in_table = float(sales.sum(axis=0)[0])
fixed = float(sales.sum(axis=0, dtype="float64")[0])

print(f"true total            {true:>18,.2f}")
print(f"sum(axis=0)           {in_table:>18,.2f}   off by {in_table - true:+,.2f}")
print(f"off by, percent       {100 * (in_table - true) / true:>+17.3f}%")
print(f'with dtype="float64"  {fixed:>18,.2f}   off by {fixed - true:+,.2f}')
del sales, price, qty

true total              1,010,107,464.38
sum(axis=0)               972,393,024.00   off by -37,714,440.38
off by, percent                  -3.734%
with dtype="float64"    1,010,107,464.38   off by +0.00


Prices are not ones, so the total doesn't freeze, but it does the same thing in a
smaller way: once the running total is large, each new price is rounded to the
coarse steps `float32` has up there. Thirty-seven million short, and still no
warning.

## Where the same call is fine

The trouble is the pairing of `float32`, a total along an axis, and that axis not
being the fast one in memory. Change any of the three and it goes away:

In [17]:
table = np.ones((N, 2), dtype="float32")
print(float(table.sum()))
print(float(table[:, 0].sum()))
f_order = np.ones((N, 2), dtype="float32", order="F")
print(f_order.sum(axis=0).tolist())
print(np.ones(3).dtype)

40000000.0
20000000.0
[20000000.0, 20000000.0]
float64


No axis at all (all forty million ones, pairwise), one column picked out, the same
table stored column by column (`order="F"`), and NumPy's default `dtype`, which is
`float64`. You only meet this once something chose `float32` for you: a file
format, a GPU library, or a "save memory" line.

`float64` has the same wall, much further out:

In [18]:
big = np.float64(2**53)
print(f"{2**53:,}")
print(big + 1 == big)

9,007,199,254,740,992
True


---

[@therestiscode](https://youtube.com/@therestiscode)